# 01 - Data Extraction and Initial Understanding

This notebook performs extraction-only work for the 30-day hospital readmission project.

**Scope**
- Load the raw dataset
- Inspect shape, columns, dtypes, and sample rows
- Build a basic data dictionary
- Save a raw snapshot summary without modifying source data

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Locate and load raw dataset

The expected source file is `data/raw/hospital_readmissions.csv`.
For compatibility with alternate dataset naming, the notebook falls back to `data/raw/diabetic_data.csv`.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

preferred_raw_path = PROJECT_ROOT / "data" / "raw" / "hospital_readmissions.csv"
fallback_raw_path = PROJECT_ROOT / "data" / "raw" / "diabetic_data.csv"

if preferred_raw_path.exists():
    raw_path = preferred_raw_path
    source_used = "preferred_hospital_readmissions"
elif fallback_raw_path.exists():
    raw_path = fallback_raw_path
    source_used = "fallback_diabetic_data"
else:
    raise FileNotFoundError(
        "Neither data/raw/hospital_readmissions.csv nor data/raw/diabetic_data.csv was found."
    )

print(f"Using raw file ({source_used}): {raw_path}")
raw_df = pd.read_csv(raw_path)
raw_df_original = raw_df.copy(deep=True)


## 2) Initial inspection

In [ ]:
print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

print("\nData types:")
display(raw_df.dtypes.to_frame("dtype"))

print("\nSample rows:")
display(raw_df.head(10))

## 3) Basic data dictionary

Data dictionary includes:
- Column name
- Data type
- Null count
- Unique value count

In [ ]:
data_dictionary_df = pd.DataFrame({
    "column_name": raw_df.columns,
    "dtype": [str(raw_df[col].dtype) for col in raw_df.columns],
    "null_count": [int(raw_df[col].isna().sum()) for col in raw_df.columns],
    "unique_values": [int(raw_df[col].nunique(dropna=False)) for col in raw_df.columns]
}).sort_values("null_count", ascending=False).reset_index(drop=True)

display(data_dictionary_df)


## 4) Save raw snapshot summary (no data mutation)

This step writes metadata artifacts only. The source dataset remains unchanged.

In [ ]:
summary_dir = PROJECT_ROOT / "reports" / "data_quality"
summary_dir.mkdir(parents=True, exist_ok=True)

summary_payload = {
    "source_file": str(raw_path),
    "shape": {"rows": int(raw_df.shape[0]), "columns": int(raw_df.shape[1])},
    "column_names": raw_df.columns.tolist(),
    "dtypes": {col: str(dtype) for col, dtype in raw_df.dtypes.items()},
    "null_count": {col: int(raw_df[col].isna().sum()) for col in raw_df.columns},
    "unique_values": {col: int(raw_df[col].nunique(dropna=False)) for col in raw_df.columns}
}

summary_output_path = summary_dir / "hospital_readmissions_extraction_snapshot.json"
dictionary_output_path = summary_dir / "hospital_readmissions_data_dictionary.csv"

pd.Series(summary_payload).to_json(summary_output_path, indent=2)
data_dictionary_df.to_csv(dictionary_output_path, index=False)

print(f"Saved summary JSON: {summary_output_path}")
print(f"Saved data dictionary CSV: {dictionary_output_path}")

# Assertion to ensure extraction notebook did not mutate loaded frame unexpectedly.
assert raw_df.equals(raw_df_original), "Raw dataframe changed during extraction."
print("Extraction integrity check passed: raw_df unchanged.")